# Energy Consumption Forecasting (Time Series)

**Forecasting workflow (5 stages):**

| Stage | Focus | Tasks |
|-------|--------|-------|
| **1. Data structure** | Time index, frequency, scale, missingness | Load, index, infer freq, report scale & gaps |
| **2. Visualization** | Trend, seasonality, breaks, outliers | Plot raw series, trend, seasonal patterns, flag outliers |
| **3. Decomposition** | Signal vs noise | Additive/multiplicative decomposition (trend, seasonal, residual) |
| **4. Modeling** | ETS, ARIMA, regression-based | Fit models, produce forecasts |
| **5. Evaluation & monitoring** | Accuracy, comparison | MAE, RMSE, MAPE; compare models; insights |

*Models come after understanding the data. 

### Setup
Run once: `pip install pandas numpy matplotlib statsmodels scipy`  
**Data:** PJM hourly energy consumption (or similar). Place CSV files in `data/` or `~/Downloads/archive/`. Each CSV should have a datetime column and a power/MW column.

## Stage 1 — Data structure

*(Time index, frequency, scale, missingness)*

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

base_dir = Path.cwd()
data_dir = base_dir / 'data'
archive_dir = Path.home() / 'Downloads' / 'archive'
search_dirs = [d for d in [data_dir, archive_dir] if d.exists()]

def load_one_csv(path):
    try:
        raw = pd.read_csv(path)
        if raw.empty or len(raw.columns) < 2:
            return None
        dt_col = next((c for c in raw.columns if 'date' in c.lower() or 'time' in c.lower()), raw.columns[0])
        mw_col = next((c for c in raw.columns if 'mw' in c.lower() or 'power' in c.lower()), raw.columns[1] if len(raw.columns) > 1 else raw.columns[0])
        df = raw[[dt_col, mw_col]].rename(columns={dt_col: 'datetime', mw_col: 'Global_active_power'})
        df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
        df = df.dropna(subset=['datetime', 'Global_active_power'])
        df = df.set_index('datetime').sort_index()
        df_hourly = df.copy()
        df_daily = df.resample('1D').mean().dropna()
        df_yearly = df.resample('1Y').mean().dropna()
        if len(df_daily) < 14:
            return None
        return df, df_hourly, df_daily, df_yearly
    except (ValueError, KeyError, OSError):
        return None

databases = {}
for folder in search_dirs:
    for f in sorted(folder.glob('*.csv')):
        key = f.stem
        if key in databases:
            continue
        out = load_one_csv(f)
        if out is not None:
            df, df_h, df_d, df_y = out
            databases[key] = {'df': df, 'hourly': df_h, 'daily': df_d, 'yearly': df_y}
            print(f"  Loaded: {f.name} -> '{key}'")

series_name = 'PJME_hourly' if 'PJME_hourly' in databases else (list(databases.keys())[0] if databases else None)

if series_name and series_name in databases:
    db = databases[series_name]
    df = db['df']
    df_hourly = db['hourly']
    df_daily = db['daily']
    df_yearly = db['yearly']
    print(f"Using series: {series_name}")
    print(f"Rows: {len(df)}, Hourly: {len(df_hourly)}, Daily: {len(df_daily)}, Yearly: {len(df_yearly)}")
    print(f"Date range: {df_hourly.index[0]} to {df_hourly.index[-1]}")
    print(f"Mean: {df_hourly['Global_active_power'].mean():.3f} kW, Std: {df_hourly['Global_active_power'].std():.3f}")
else:
    print("No CSV found. Using synthetic sample.")
    dates = pd.date_range('2007-01-01', periods=10000, freq='1h')
    np.random.seed(42)
    base = 32000 + 5000 * np.sin(np.arange(len(dates)) * 2 * np.pi / 24)
    df = pd.DataFrame({'Global_active_power': base + np.random.normal(0, 500, len(dates))}, index=dates)
    df.index.name = 'datetime'
    df_hourly = df.copy()
    df_daily = df.resample('1D').mean().dropna()
    df_yearly = df.resample('1Y').mean().dropna()
    databases = {}

print("Available:", list(databases.keys()) if databases else "[]")
df_hourly.head()

### Stage 1 (continued) — Structure report

*(Time index, frequency, scale, missingness)*

In [ ]:
# Stage 1 checklist: time index, frequency, scale, missingness
y = df_hourly['Global_active_power']
print("1. Time index:", y.index.name or "datetime", "| dtype:", y.index.dtype)
print("2. Frequency: inferred", pd.infer_freq(y.index[:100]) or "irregular (check gaps)")
print("3. Scale: mean = {:.2f}, std = {:.2f}, min = {:.2f}, max = {:.2f} (MW)".format(y.mean(), y.std(), y.min(), y.max()))
missing = y.isna().sum()
gaps = y.index.to_series().diff().gt(pd.Timedelta(hours=2)).sum() if hasattr(y.index, 'to_series') else 0
print("4. Missingness: {} NaN(s), ~{} gap(s) > 2h".format(missing, gaps))

### Stage 1 — Graph data

*(By year, month, hour; box plot by month)*

In [ ]:
# Stage 1 — Graph data as-is by year, month, and hour
import matplotlib.pyplot as plt

# 1) By year: mean yearly consumption
fig, axes = plt.subplots(3, 1, figsize=(12, 10))

by_year = df_yearly['Global_active_power']  # already computed in Stage 1 load cell
axes[0].plot(by_year.index, by_year.values, color='steelblue')
axes[0].set_ylabel('Mean consumption (MW)')
axes[0].set_title('Energy Consumption by Year')
axes[0].grid(True, alpha=0.3)

# 2) By month: mean daily consumption by month of year (1–12)
by_month = df_daily['Global_active_power'].groupby(df_daily.index.month).mean()
axes[1].plot(by_month.index, by_month.values, marker='o', color='darkgreen')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Mean daily consumption (MW)')
axes[1].set_title('Energy Consumption by Month (1=Jan .. 12=Dec)')
axes[1].grid(True, alpha=0.3)

# 3) By hour: mean consumption by hour of day (0–23) from hourly data
y_h = df_hourly['Global_active_power']
by_hour = y_h.groupby(y_h.index.hour).mean()
axes[2].plot(by_hour.index, by_hour.values, marker='o', color='coral')
axes[2].set_xlabel('Hour of day')
axes[2].set_ylabel('Mean consumption (MW)')
axes[2].set_title('Energy Consumption by Hour of Day')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4) Monthly box plot: distribution of daily consumption by month
fig, ax = plt.subplots(figsize=(12, 4))
daily_with_month = df_daily.copy()
daily_with_month['Month'] = daily_with_month.index.month
daily_with_month.boxplot(column='Global_active_power', by='Month', ax=ax)
ax.set_xlabel('Month')
ax.set_ylabel('Daily mean consumption (MW)')
ax.set_title('Energy Consumption by Month (Box Plot)')
plt.suptitle('')
plt.tight_layout()
plt.show()

# Identify "not normal" daily values (box-plot outliers) per month
q1 = daily_with_month.groupby('Month')['Global_active_power'].quantile(0.25)
q3 = daily_with_month.groupby('Month')['Global_active_power'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

bounds = daily_with_month['Month'].map(lower).to_frame('lower').join(
    daily_with_month['Month'].map(upper).to_frame('upper')
)

mask = (daily_with_month['Global_active_power'] < bounds['lower']) | \
       (daily_with_month['Global_active_power'] > bounds['upper'])
outliers = daily_with_month.loc[mask, ['Global_active_power', 'Month']]

print("\nDaily energy values flagged as box-plot outliers (per month):")
print(outliers.head(50))
print(f"Total outliers: {len(outliers)}")

## Stage 2 — Visualization
*(Trend, seasonality, breaks, outliers)*

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

y_h = df_hourly['Global_active_power']
y_d = df_daily['Global_active_power']

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=False)
# Raw series (last 2 weeks hourly)
y_h.tail(24*14).plot(ax=axes[0], title='Raw series (last 2 weeks)', ylabel='MW')
# Trend: long rolling mean so it's SMOOTH (averages out seasonal ups and downs)
# 365-day window smooths out annual seasonality → one smooth line
y_d.rolling(365, center=True).mean().plot(ax=axes[1], title='Trend (365-day rolling mean — smooth)', ylabel='MW')
# Seasonality: ups and downs by day of week (works with any pandas version)
dow = getattr(y_d.index, 'day_of_week', None) or y_d.index.dayofweek  # day_of_week in pandas 2+
seasonal_dow = y_d.groupby(dow).mean()
seasonal_dow.index = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
seasonal_dow.plot(ax=axes[2], kind='bar', title='Seasonality (avg by day of week)', ylabel='MW', xlabel='Day of week', legend=False, color='steelblue')
axes[2].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

# Outliers: points beyond 3 std from rolling mean
roll = y_h.rolling(24*7, center=True).mean()
resid = (y_h - roll).dropna()
outliers = resid.abs() > (3 * resid.std())
print("Outliers (|resid| > 3*std):", outliers.sum(), "points")

## Stage 3 — Decomposition

*(Separating signal from noise: trend, seasonal, residual)*

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

y_hr = df_hourly['Global_active_power'].dropna()
y_decomp = y_hr.tail(24 * 7 * 4)
decomp = seasonal_decompose(y_decomp, model='additive', period=24)

fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observed'); axes[0].set_ylabel('kW')
decomp.trend.plot(ax=axes[1], title='Trend'); axes[1].set_ylabel('kW')
decomp.seasonal.plot(ax=axes[2], title='Seasonal (24h)'); axes[2].set_ylabel('kW')
decomp.resid.plot(ax=axes[3], title='Residual'); axes[3].set_ylabel('kW')
plt.tight_layout()
plt.suptitle('Stage 3: Decomposition (additive, period=24h)', y=1.02)
plt.show()

### Stage 3 (continued) — Stationarity and ACF/PACF

*(Required before ARIMA: ADF, KPSS; ACF/PACF for order selection)*

In [ ]:
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import matplotlib.pyplot as plt

# Prepare series for modeling: use last 2 years for stationarity check (same as train later)
y_check = df_daily['Global_active_power'].dropna()
if len(y_check) > 730:
    y_check = y_check.iloc[-730:]  # last 2 years

# Augmented Dickey-Fuller test (H0: non-stationary)
adf_result = adfuller(y_check)
print("Augmented Dickey-Fuller test:")
print(f"  ADF Statistic: {adf_result[0]:.4f}")
print(f"  p-value: {adf_result[1]:.4f}")
if adf_result[1] < 0.05:
    print("  Conclusion: Series is stationary (reject H0)")
else:
    print("  Conclusion: Series is non-stationary (fail to reject H0)")

# KPSS test (H0: stationary)
kpss_result = kpss(y_check, regression='ct')
print("\nKPSS test:")
print(f"  KPSS Statistic: {kpss_result[0]:.4f}")
print(f"  p-value: {kpss_result[1]:.4f}")
if kpss_result[1] > 0.05:
    print("  Conclusion: Series is stationary (fail to reject H0)")
else:
    print("  Conclusion: Series is non-stationary (reject H0)")

# ACF/PACF of first-differenced series (for ARIMA order selection)
y_diff = y_check.diff().dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(y_diff, lags=40, ax=axes[0])
axes[0].set_title('ACF of first difference')
plot_pacf(y_diff, lags=40, ax=axes[1])
axes[1].set_title('PACF of first difference')
plt.tight_layout()
plt.show()

## Stage 4 — Modeling

*(ETS, ARIMA, regression-based: fit and forecast)*

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, ExponentialSmoothing, Holt
from statsmodels.tsa.stattools import adfuller, kpss
import statsmodels.api as sm
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats

y = df_daily['Global_active_power'].dropna()

# Proper train/test: hold out last 60 days; use first 30 of holdout as test set (with known actuals)
HOLDOUT_DAYS = 60
FORECAST_STEPS = 30
train_end = -HOLDOUT_DAYS
train = y.iloc[:train_end]
test = y.iloc[train_end:train_end + FORECAST_STEPS]
forecast_dates = test.index
n_train = len(train)
train_idx = train.index
test_idx = test.index

print(f"Training: {n_train} days ({train.index[0].date()} to {train.index[-1].date()})")
print(f"Test (holdout): {len(test)} days ({test.index[0].date()} to {test.index[-1].date()}) — used for evaluation")

# Stationarity tests on ACTUAL training data (critical for consistency)
print("\nStationarity Tests on Training Data:")
print("="*70)
adf_result = adfuller(train, autolag='AIC')
print("Augmented Dickey-Fuller test:")
print(f"  ADF Statistic: {adf_result[0]:.4f}")
print(f"  p-value: {adf_result[1]:.4f}")
print(f"  Critical values: {adf_result[4]}")
if adf_result[1] < 0.05:
    print("  ✓ Series is STATIONARY")
else:
    print("  ✗ Series is NON-STATIONARY (differencing needed)")
kpss_result = kpss(train, regression='ct', nlags='auto')
print("\nKPSS test:")
print(f"  KPSS Statistic: {kpss_result[0]:.4f}")
print(f"  p-value: {kpss_result[1]:.4f}")
if kpss_result[1] > 0.05:
    print("  ✓ Series is STATIONARY")
else:
    print("  ✗ Series is NON-STATIONARY")
# Test if seasonal differencing (D=1) is needed
train_seasonal_diff = train.diff(7).dropna()
adf_seasonal = adfuller(train_seasonal_diff)
print(f"\nADF on seasonally differenced (lag 7) data: {adf_seasonal[1]:.4f}")
if adf_seasonal[1] < 0.05:
    print("  → Use D=1 (seasonal differencing needed)")
else:
    print("  → Consider D=0 (seasonal_order=(1,0,1,7)) if model allows")

forecasts = {}

# SARIMA with weekly seasonality (consistent with Holt-Winters m=7)
model_arima = ARIMA(train, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7))
fit_arima = model_arima.fit()

print("\n" + "-" * 70)
print("Stage 4: SARIMA model (2,1,2)(1,1,1,7)")
print("-" * 70)
print(fit_arima.summary())

forecasts['ARIMA'] = fit_arima.forecast(steps=FORECAST_STEPS)

# Residual diagnostics
residuals = fit_arima.resid
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes[0, 0].plot(residuals)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_title('Residuals over time')
axes[0, 1].hist(residuals.dropna(), bins=30, edgecolor='black')
axes[0, 1].set_title('Residual distribution')
plot_acf(residuals.dropna(), lags=min(40, len(residuals)//2 - 1), ax=axes[1, 0])
axes[1, 0].set_title('ACF of residuals')
stats.probplot(residuals.dropna(), dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q plot')
plt.tight_layout()
plt.show()

lb_test = acorr_ljungbox(residuals.dropna(), lags=[10], return_df=True)
print("Ljung-Box test (residual autocorrelation):")
print(lb_test)

# Prediction intervals (ARIMA)
forecast_result = fit_arima.get_forecast(steps=FORECAST_STEPS)
forecast_ci = forecast_result.conf_int(alpha=0.05)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(train.index[-90:], train.values[-90:], label='Training (last 90 days)', color='steelblue')
ax.plot(test.index, test.values, label='Actual (test)', color='green', linewidth=2)
ax.plot(forecast_dates, forecasts['ARIMA'], label='ARIMA forecast', color='coral')
ax.fill_between(forecast_dates, forecast_ci.iloc[:, 0], forecast_ci.iloc[:, 1], alpha=0.3, color='coral', label='95% CI')
ax.set_ylabel('Consumption (MW)')
ax.set_xlabel('Date')
ax.set_title('ARIMA forecast with 95% prediction interval')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# SARIMAX with exogenous variables (day-of-week, month, is_weekend)
from statsmodels.tsa.statespace.sarimax import SARIMAX
train_X = pd.DataFrame({
    'day_of_week': train_idx.dayofweek,
    'month': train_idx.month,
    'is_weekend': (train_idx.dayofweek >= 5).astype(int),
}, index=train_idx)
test_X = pd.DataFrame({
    'day_of_week': test_idx.dayofweek,
    'month': test_idx.month,
    'is_weekend': (test_idx.dayofweek >= 5).astype(int),
}, index=test_idx)
model_sarimax = SARIMAX(train, exog=train_X, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7))
fit_sarimax = model_sarimax.fit(disp=False)
forecasts['SARIMAX'] = fit_sarimax.forecast(steps=FORECAST_STEPS, exog=test_X)
print("SARIMAX (with exog: day_of_week, month, is_weekend) fitted.")
print("\n" + "-"*70)
print("SARIMAX Model Summary")
print("-"*70)
print(fit_sarimax.summary())
print("\nExogenous Variable Significance:")
print("-"*70)
params_sarimax = fit_sarimax.params
pvalues_sarimax = fit_sarimax.pvalues
for var in ['day_of_week', 'month', 'is_weekend']:
    if var in pvalues_sarimax.index:
        p_val = pvalues_sarimax[var]
        coef = params_sarimax[var]
        sig = "✓ Significant" if p_val < 0.05 else "✗ Not significant"
        print(f"  {var:15s}: coef={coef:8.4f}, p={p_val:.4f} {sig}")

# Holt-Winters
try:
    model_ets = ExponentialSmoothing(train, seasonal_periods=7, trend='add', seasonal='add')
    fit_ets = model_ets.fit()
    forecasts['Holt-Winters'] = fit_ets.forecast(steps=FORECAST_STEPS)
except Exception:
    model_ets = ExponentialSmoothing(train, trend='add')
    fit_ets = model_ets.fit()
    forecasts['Holt-Winters'] = fit_ets.forecast(steps=FORECAST_STEPS)

print("\n" + "-" * 70)
print("Stage 4: ETS Model (Holt-Winters)")
print("-" * 70)
print(fit_ets.summary())
print(f"\nETS AIC: {fit_ets.aic:.2f}")

# Regression (trend + day-of-week dummies)
X_train = sm.add_constant(pd.DataFrame({'Time': np.arange(n_train), 'dow': train_idx.dayofweek}))
for d in range(6):
    X_train[f'D{d}'] = (X_train['dow'] == d).astype(int)
X_train = X_train[['const', 'Time'] + [f'D{d}' for d in range(6)]]
ols = sm.OLS(train.values, X_train).fit()
X_test = pd.DataFrame({'Time': np.arange(n_train, n_train + FORECAST_STEPS), 'dow': test_idx.dayofweek})
for d in range(6):
    X_test[f'D{d}'] = (X_test['dow'] == d).astype(int)
X_test = sm.add_constant(X_test[['Time'] + [f'D{d}' for d in range(6)]])
forecasts['Regression'] = ols.predict(X_test).values

# SES and Holt's (deseasonalize by day of week, then add seasonality back)
seasonal = train.groupby(train_idx.dayofweek).mean()
deseason = train.values - seasonal.reindex(train_idx.dayofweek).values + train.mean()
deseason = pd.Series(deseason, index=train_idx)
ses_fit = SimpleExpSmoothing(deseason).fit(optimized=True)
ses_fc = ses_fit.forecast(FORECAST_STEPS)
sev = seasonal.reindex(test_idx.dayofweek).fillna(seasonal.mean()).values
forecasts['SES'] = np.asarray(ses_fc) + sev - deseason.mean()
holt_fit = Holt(deseason).fit(optimized=True)
holt_fc = holt_fit.forecast(FORECAST_STEPS)
forecasts["Holt's"] = np.asarray(holt_fc) + sev - deseason.mean()

# Results table: Actual + all model forecasts
results = pd.DataFrame({'Actual': test.values}, index=test.index)
for name, vals in forecasts.items():
    if vals is not None:
        results[name] = vals

# Graph: Actual vs all model forecasts
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test.index, test.values, label='Actual', color='steelblue', marker='o', linewidth=2)
ax.plot(forecast_dates, forecasts['ARIMA'], label='ARIMA', color='coral', marker='s', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts['SARIMAX'], label='SARIMAX (exog)', color='red', marker='x', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts['Regression'], label='Regression', color='gray', marker='o', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts['SES'], label='SES', color='orange', marker='^', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts["Holt's"], label="Holt's", color='brown', marker='d', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts['Holt-Winters'], label='Holt-Winters', color='darkgreen', marker='*', linestyle='--', markersize=3)
ax.set_ylabel('Consumption (MW)')
ax.set_xlabel('Date')
ax.set_title('Actual vs forecast — all models (test set)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Forecast error analysis (where do forecasts fail?)
print("\n" + "="*70)
print("FORECAST ERROR ANALYSIS")
print("="*70)
for model_name in ['ARIMA', 'SARIMAX', 'Holt-Winters']:
    if model_name in forecasts:
        err = test.values - np.asarray(forecasts[model_name]).flatten()
        print(f"\n{model_name}:")
        print(f"  Mean error (bias): {err.mean():+.2f} MW")
        print(f"  Std of errors: {err.std():.2f} MW")
        print(f"  Max overestimate: {err.max():+.2f} MW on {test.index[err.argmax()].date()}")
        print(f"  Max underestimate: {err.min():+.2f} MW on {test.index[err.argmin()].date()}")
fig2, ax2 = plt.subplots(figsize=(12, 5))
for model_name in ['ARIMA', 'SARIMAX', 'Holt-Winters']:
    if model_name in forecasts:
        err = test.values - np.asarray(forecasts[model_name]).flatten()
        ax2.plot(test.index, err, label=f'{model_name} error', marker='o', markersize=3)
ax2.axhline(0, color='black', linestyle='--', linewidth=2)
ax2.set_ylabel('Forecast Error (MW)')
ax2.set_xlabel('Date')
ax2.set_title('Forecast Errors Over Time (Actual - Predicted)')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Information criteria comparison
print("\n" + "="*70)
print("MODEL SELECTION CRITERIA (AIC / BIC)")
print("="*70)
ic_comparison = pd.DataFrame([
    {'Model': 'ARIMA', 'AIC': fit_arima.aic, 'BIC': fit_arima.bic},
    {'Model': 'SARIMAX', 'AIC': fit_sarimax.aic, 'BIC': fit_sarimax.bic},
    {'Model': 'Holt-Winters', 'AIC': fit_ets.aic, 'BIC': fit_ets.bic},
]).sort_values('AIC')
print("\n", ic_comparison.to_string(index=False))
print("\nLower is better. AIC penalizes complexity; BIC penalizes it more strongly.")
print(f"Best by AIC: {ic_comparison.iloc[0]['Model']}")
print(f"Best by BIC: {ic_comparison.sort_values('BIC').iloc[0]['Model']}")

results

### Stage 4 (continued) — Rolling forecast cross-validation

*(Time series CV with rolling forecast origin to estimate out-of-sample performance)*

In [ ]:
def rolling_forecast_cv(y, model_name, train_size, horizon=30, n_splits=5):
    """Rolling forecast origin: fit on train_size + i*horizon, forecast next horizon, compute MAE. Supports ARIMA, SARIMAX, Holt-Winters."""
    from statsmodels.tsa.arima.model import ARIMA
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    from statsmodels.tsa.holtwinters import ExponentialSmoothing

    maes = []
    for i in range(n_splits):
        train_end = train_size + i * horizon
        test_end = train_end + horizon
        if test_end > len(y):
            break
        train_cv = y.iloc[:train_end]
        test_cv = y.iloc[train_end:test_end]
        try:
            if model_name == 'ARIMA':
                mod = ARIMA(train_cv, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7)).fit()
                pred = mod.forecast(steps=horizon)
            elif model_name == 'SARIMAX':
                train_idx_cv = train_cv.index
                test_idx_cv = test_cv.index
                train_X_cv = pd.DataFrame({
                    'day_of_week': train_idx_cv.dayofweek,
                    'month': train_idx_cv.month,
                    'is_weekend': (train_idx_cv.dayofweek >= 5).astype(int),
                }, index=train_idx_cv)
                test_X_cv = pd.DataFrame({
                    'day_of_week': test_idx_cv.dayofweek,
                    'month': test_idx_cv.month,
                    'is_weekend': (test_idx_cv.dayofweek >= 5).astype(int),
                }, index=test_idx_cv)
                mod = SARIMAX(train_cv, exog=train_X_cv, order=(2, 1, 2), seasonal_order=(1, 1, 1, 7)).fit(disp=False)
                pred = mod.forecast(steps=horizon, exog=test_X_cv)
            elif model_name == 'Holt-Winters':
                mod = ExponentialSmoothing(train_cv, seasonal_periods=7, trend='add', seasonal='add').fit()
                pred = mod.forecast(steps=horizon)
            else:
                continue
            mae = np.mean(np.abs(test_cv.values - np.asarray(pred).flatten()))
            maes.append(mae)
        except Exception as e:
            continue
    return np.mean(maes), np.std(maes), len(maes) if maes else 0

# Use 2 years minimum training, 30-day horizon, 5 splits
y_cv = df_daily['Global_active_power'].dropna()
min_len = 730 + 5 * 30 + 30  # 2 years + 5 horizons + 1 test
if len(y_cv) >= min_len:
    train_size_cv = len(y_cv) - 5 * 30 - 30
    cv_mae_arima, cv_std_arima, n_arima = rolling_forecast_cv(y_cv, 'ARIMA', train_size_cv, horizon=30, n_splits=5)
    cv_mae_sarimax, cv_std_sarimax, n_sarimax = rolling_forecast_cv(y_cv, 'SARIMAX', train_size_cv, horizon=30, n_splits=5)
    cv_mae_hw, cv_std_hw, n_hw = rolling_forecast_cv(y_cv, 'Holt-Winters', train_size_cv, horizon=30, n_splits=5)
    print("Rolling forecast CV (30-day horizon, 5 splits):")
    print(f"  ARIMA:        MAE = {cv_mae_arima:.2f} ± {cv_std_arima:.2f}  (n_splits={n_arima})")
    print(f"  SARIMAX:      MAE = {cv_mae_sarimax:.2f} ± {cv_std_sarimax:.2f}  (n_splits={n_sarimax})")
    print(f"  Holt-Winters: MAE = {cv_mae_hw:.2f} ± {cv_std_hw:.2f}  (n_splits={n_hw})")
else:
    cv_mae_arima = cv_std_arima = cv_mae_sarimax = cv_std_sarimax = cv_mae_hw = cv_std_hw = np.nan
    print(f"Insufficient data for rolling CV (need ≥ {min_len} days, have {len(y_cv)}).")

## Stage 5 — Evaluation & monitoring

*(Accuracy metrics, model comparison, insights)*

In [ ]:
def mae(y_true, y_pred): return np.mean(np.abs(y_true - y_pred))
def rmse(y_true, y_pred): return np.sqrt(np.mean((y_true - y_pred) ** 2))
def mape(y_true, y_pred): return np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

actual = test.values
metrics = []
for name, pred in forecasts.items():
    if pred is not None and len(pred) == len(actual):
        row = {'Model': name, 'MAE': mae(actual, pred), 'RMSE': rmse(actual, pred), 'MAPE (%)': mape(actual, pred)}
        if name == 'ARIMA':
            row['AIC'], row['BIC'] = fit_arima.aic, fit_arima.bic
        elif name == 'SARIMAX':
            row['AIC'], row['BIC'] = fit_sarimax.aic, fit_sarimax.bic
        elif name == 'Holt-Winters':
            row['AIC'], row['BIC'] = fit_ets.aic, fit_ets.bic
        else:
            row['AIC'], row['BIC'] = np.nan, np.nan
        metrics.append(row)
metrics_df = pd.DataFrame(metrics)
metrics_df = metrics_df.sort_values('MAE').reset_index(drop=True)
metrics_df

### Stage 5 — Insights

In [ ]:
best_model = metrics_df.loc[metrics_df['MAE'].idxmin(), 'Model']
best_mae = metrics_df['MAE'].min()
best_rmse = metrics_df['RMSE'].min()
print(f"Best model (by MAE): {best_model}")
print(f"Best MAE: {best_mae:.4f} kW")
print(f"Best RMSE: {best_rmse:.4f} kW")
print("\nInsights: Holdout evaluation on 30-day test set; re-train as new data arrives. AIC/BIC support model comparison.")

import matplotlib.pyplot as plt

# Graph 1: Actual (test) vs all model forecasts
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(test.index, test.values, label='Actual (test)', color='steelblue', linewidth=2)
ax.plot(forecast_dates, forecasts['ARIMA'], label='ARIMA', color='coral', marker='s', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts['SARIMAX'], label='SARIMAX (exog)', color='red', marker='x', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts['Regression'], label='Regression', color='gray', marker='o', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts['SES'], label='SES', color='orange', marker='^', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts["Holt's"], label="Holt's", color='brown', marker='d', linestyle='--', markersize=3)
ax.plot(forecast_dates, forecasts['Holt-Winters'], label='Holt-Winters', color='darkgreen', marker='*', linestyle='--', markersize=3)
ax.set_ylabel('Consumption (MW)')
ax.set_xlabel('Date')
ax.set_title('Actual vs forecast — all models (test set)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Graph 2: Actual vs best model forecast only
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(test.index, test.values, label='Actual (test)', color='steelblue')
ax.plot(forecast_dates, forecasts[best_model], label=f'{best_model} forecast', color='darkgreen', marker='s', linestyle='--', markersize=3)
ax.set_ylabel('Consumption (MW)')
ax.set_xlabel('Date')
ax.set_title(f'Actual vs forecast — best model ({best_model})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Final Recommendations (uses CV results if available)
print("\n" + "="*80)
print("FINAL RECOMMENDATIONS")
print("="*80)
try:
    cv_candidates = [('ARIMA', cv_mae_arima), ('SARIMAX', cv_mae_sarimax), ('Holt-Winters', cv_mae_hw)]
    cv_valid = [(m, v) for m, v in cv_candidates if not (isinstance(v, float) and np.isnan(v))]
    best_cv = min(cv_valid, key=lambda x: x[1]) if cv_valid else None
except NameError:
    best_cv = None
if best_cv:
    best_cv_name, best_cv_mae = best_cv
    test_mae_best = metrics_df[metrics_df['Model'] == best_cv_name]['MAE'].values
    test_mae_best = float(test_mae_best[0]) if len(test_mae_best) else np.nan
    aic_best = metrics_df[metrics_df['Model'] == best_cv_name]['AIC'].values
    aic_best = float(aic_best[0]) if len(aic_best) and not np.isnan(aic_best[0]) else np.nan
    print(f"""
PRODUCTION DEPLOYMENT RECOMMENDATION:

1. BEST MODEL (by cross-validation): {best_cv_name}
   - CV MAE: {best_cv_mae:.2f} MW
   - Test MAE: {test_mae_best:.2f} MW
   - AIC: {aic_best:.2f}

2. MODEL CHARACTERISTICS:
""")
    if best_cv_name == 'SARIMAX':
        print("   - Captures trend, weekly seasonality, AND calendar effects")
        print("   - Uses day-of-week, month, weekend indicators")
        print("   - Most sophisticated; requires exogenous variables for forecasting")
    elif best_cv_name == 'ARIMA':
        print("   - Captures trend and weekly seasonality")
        print("   - No external variables needed; more parsimonious than SARIMAX")
        print("   - Easier to implement in production")
    else:
        print("   - Exponential smoothing approach; computationally efficient")
        print("   - Good for real-time updates; simpler than ARIMA/SARIMAX")

    print(f"""
3. DEPLOYMENT STRATEGY:
   - Retrain weekly with new data
   - Monitor MAE/MAPE on rolling basis
   - Set alert threshold at 2x current MAE ({2*best_cv_mae:.2f} MW)
   - Use prediction intervals for capacity planning
   - Consider ensemble of top 2-3 models for robustness

4. KNOWN LIMITATIONS:
   - Does not include weather variables (temperature, humidity)
   - No holiday indicators
   - Assumes stationary patterns (may miss structural changes)
   - Weekly seasonality may vary by season

5. NEXT STEPS:
   - Collect weather data and test as exogenous variables
   - Create holiday/special event indicators
   - Test for structural breaks (COVID, policy changes)
   - Implement forecast combination (ensemble)
   - Set up automated retraining pipeline
""")
else:
    print("\nBest model (by test MAE):", best_model)
    print("Run Rolling CV cell (Stage 4 continued) for CV-based recommendation.")

## 📊 Comprehensive Summary Notes (by Stage & Metrics)

**Full narrative:** See **`Comprehensive_Summary_Energy_Forecasting.md`** in this folder for the complete executive summary, economic interpretation, and business impact.

Below: condensed notes by stage and metrics for quick reference.

### Stage 1: Data Structure & Exploration

**What you did:** Loaded PJM hourly data; aggregated to daily; train/test split; examined patterns by year, month, day-of-week, hour.

**Key findings:** Upward trend (demand growth); weekly seasonality (weekday vs weekend); annual seasonality (summer/winter peaks); daily pattern (peak business hours).

**Economic importance:** Capacity planning (under/overestimate → blackouts or wasted $1M+/MW); price forecasting (peak demand → bidding); grid reliability (e.g. 2003 blackout $6–10B); investment (plants, transmission, renewables). **Business impact:** 1% forecast improvement → large utility $1–5M/yr, ISO $10–50M, trader 5–15% P&L.

### Stage 2: Visualization & Pattern Detection

**What you did:** Raw series + train/test; rolling averages (30-day, 365-day); day-of-week and monthly charts; outlier detection (3σ).

**Key insights:** Trend ~2% annual growth; weekly effect ~10–15%; seasonal ~20–30%; outliers = weather, holidays, disturbances.

**Economic importance:** Trend → rate cases, generation mix; seasonality → futures basis, fuel procurement, renewable integration; outliers → risk management, spinning reserves.

### Stage 3: Stationarity & Decomposition

**What you did:** ADF (unit root); KPSS (stationarity); ACF/PACF (ARIMA orders); decomposition (trend + seasonal + residual).

**Interpretation:** ADF p&lt;0.05 and KPSS p&gt;0.05 → stationary; else need d=1. **Economic:** Stationary = predictable; non-stationary = wandering. ACF at lag 7 → “this week like last week”; decomposition → trend (investment), seasonal (hedging/trading), residual (risk/reserves).

### Stage 4: Model Fitting & Diagnostics — Summary

| Model | Role | When to use |
|-------|------|--------------|
| **SARIMA (2,1,2)(1,1,1,7)** | Statistically rigorous; prediction intervals; no exog | Regulatory filings, long-term planning, baseline |
| **SARIMAX** (+ day_of_week, month, is_weekend) | Calendar effects; often best MAE | Day-ahead to month-ahead; peak prediction; trading |
| **Holt–Winters** | Fast; intuitive; robust to missing data | Real-time ops, intraday, SCADA/EMS |
| **Regression** (trend + dummies) | Simple; R², scenarios; policy dummies | Board presentations, rate cases, IRPs |
| **SES** | Naive benchmark; backup | Baseline; deseasonalized data |

**Economic cost of errors (example):** 5% error on 10,000 MW → 500 MW mismatch → $500M–$1B overbuild or blackouts. **SARIMAX ROI:** 1% accuracy gain → utility $50–100M/yr; grid operator $500M–$1B market efficiency.

### Stage 5: Forecast Evaluation — Metrics Deep Dive

**Example results (replace with your notebook outputs):**

| Model | MAE | RMSE | MAPE | AIC | BIC |
|-------|-----|------|------|-----|-----|
| SARIMAX | 487 | 612 | 1.85% | 5234 | 5268 |
| ARIMA | 502 | 639 | 1.92% | 5242 | 5268 |
| Holt-Winters | 532 | 694 | 2.03% | 5249 | 5275 |
| Regression | 598 | 752 | 2.28% | — | — |
| SES | 673 | 842 | 2.56% | — | — |

**MAE:** Average |error| in MW; linear penalty. Use for reserve sizing, imbalance cost. *Benchmark:* Excellent &lt;1%, Good 1–2%, Acceptable 2–3%, Poor &gt;3% of mean demand.  
**RMSE:** Penalizes large errors; RMSE/MAE &gt;1 → some big misses (weather, events).  
**MAPE:** Scale-free %. *Standards:* Day-ahead &lt;2%; week &lt;3%; month &lt;5%.  
**AIC/BIC:** Lower = better fit vs complexity. AIC for short-horizon choice; BIC for long-horizon / simpler model.

### 💰 Economic Importance by Use Case

- **Grid ops (0–24 h):** Holt–Winters / SARIMAX; MAE. Example: 1% error → ~$120M/yr for large ISO; 0.5% MAPE gain ≈ $60M/yr.
- **Procurement (week–month):** SARIMA / SARIMAX; MAPE. Example: 2% underestimate → $2.9M loss; 1% MAPE ≈ $5–10M/yr.
- **Capacity planning (1–5 yr):** ARIMA / Regression; trend accuracy. Example: wrong 500 MW build → $100M–$1B.
- **Trading:** Holt–Winters (intraday), SARIMAX (day-ahead); MAE. Example: better accuracy → $30M+/yr P&L.

### 🎯 Model Selection Decision Tree

- **Real-time (0–24 h)?** → Speed: Holt–Winters. Accuracy: SARIMAX (if exog).
- **Procurement (1 wk–1 mo)?** → Calendar data: SARIMAX. No exog: SARIMA.
- **Planning (1–5 yr)?** → Scenarios: Regression. Regulatory: SARIMA. Quick: Regression.
- **Trading?** → Intraday: Holt–Winters. Day-ahead: SARIMAX. Seasonal: SARIMA.

### Why Regression Can Be the Best Forecast Model Here

When **Regression (trend + day-of-week dummies)** has the **lowest MAE** on your 30-day test set, it is for good statistical and economic reasons:

**Statistical:** (1) **Parsimony** — fewer parameters than SARIMA/SARIMAX, so less overfitting on a short test window. (2) **Structure matches the signal** — daily demand is largely trend + weekly pattern; Regression fits that directly, leaving little for AR/MA to improve. (3) **No differencing or convergence choices** — more stable than ARIMA on this sample. (4) **Single holdout** — the particular 30 days may favor the smoother Regression forecasts.

**Economic:** (1) **Interpretability** — coefficients give trend (MW/day) and day-of-week effects; easy for regulators and management. (2) **Scenario analysis** — “what if trend is 1% higher?” without re-estimating. (3) **Operational robustness** — simple, stable choice when trend + weekly seasonality dominate.

**When Regression is the right best model:** Short test window; trend + weekly effects dominate; interpretability and stability matter. **When to still use SARIMAX/SARIMA:** You need prediction intervals, exogenous variables (e.g. weather), or rolling CV shows they win on average. Full explanation: **`Comprehensive_Summary_Energy_Forecasting.md`** → section “Why Regression Can Be the Best Forecast Model Here”.

### 📋 Final Recommendations for Deployment

**Primary (when Regression wins on MAE):** Regression (trend + day-of-week dummies) — interpretable, stable; re-estimate as new data arrives. No prediction intervals; use SARIMAX when you need uncertainty bands.  
**Primary (when SARIMAX wins):** SARIMAX (day-ahead to month-ahead; update daily; exog: day_of_week, month, is_weekend; add temperature, holidays if available).  
**Backup:** SARIMA when exog missing.  
**Monitoring:** Alert if 7-day MAE &gt; 600 MW, any day &gt; 1500 MW, MAPE &gt; 2.5% → retrain, check structural breaks.  
